# Bibliotecas


In [13]:
import sys
from pathlib import Path

import pandas as pd
import requests
from shapely.geometry import shape

ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'config.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC_DIR = ROOT / 'src'
DATA_DIR = ROOT / 'data'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import COLECOES_RECOMENDADAS, ICECHUNK_PATH, STAC_URL, VARIAVEIS_PRIORITARIAS_POR_COLECAO
from geoparquet import itens_para_geodataframe, salvar_geoparquet
from pipeline import executar_pipeline, validar_cubo
from stac import buscar_itens, conectar_stac
from storage import abrir_repositorio, salvar_dataset_virtual, criar_repositorio

print('Projeto:', ROOT)
print('Dados:', DATA_DIR)


Projeto: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_bdgeo\trab_final_bdgeo
Dados: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_bdgeo\trab_final_bdgeo\data


## Configuração


In [2]:
CODIGO_IBGE_MUNICIPIO = '3549904'
COLECAO = 'S2-16D-2'
VARIAVEIS = VARIAVEIS_PRIORITARIAS_POR_COLECAO[COLECAO]
ICECHUNK_REPO = ROOT / 'data' / 'icechunk_repo'


## Área de estudo


In [3]:
url_malha = f'https://servicodados.ibge.gov.br/api/v3/malhas/municipios/{CODIGO_IBGE_MUNICIPIO}?formato=application/vnd.geo+json&qualidade=minima'
resposta = requests.get(url_malha, timeout=60)
resposta.raise_for_status()
MUNICIPIO = shape(resposta.json()['features'][0]['geometry'])
print('Área de estudo: São José dos Campos')

Área de estudo: São José dos Campos


## Período


In [4]:
DATA_FIM = pd.Timestamp.today().normalize()
DATA_INICIO = DATA_FIM - pd.DateOffset(years=10)
print(f'Período: {DATA_INICIO.date()} - {DATA_FIM.date()}')

Período: 2016-09-05 - 2026-09-05


## Consulta STAC


In [5]:
catalogo = conectar_stac()
cenas = buscar_itens(catalogo, colecao=COLECAO, intersects=MUNICIPIO.__geo_interface__, data_inicio=DATA_INICIO.date().isoformat(), data_fim=DATA_FIM.date().isoformat(), max_itens=None)
print(f'Cenas encontradas: {len(cenas)}')

Cenas encontradas: 222


In [6]:
if COLECAO not in COLECOES_RECOMENDADAS:
    raise ValueError(f'Coleção não configurada: {COLECAO}')
print('Coleção:', COLECAO)
print('Variáveis solicitadas:', len(VARIAVEIS))

Coleção: S2-16D-2
Variáveis solicitadas: 5


## Variáveis disponíveis


In [7]:
variaveis_configuradas = (VARIAVEIS_PRIORITARIAS_POR_COLECAO[COLECAO])
variaveis_disponiveis = tuple(variavel for variavel in variaveis_configuradas if any(variavel in cena.assets for cena in cenas))

print("Variáveis disponíveis:")
for variavel in variaveis_disponiveis:
    print(" -", variavel)

Variáveis disponíveis:
 - B01
 - B02
 - B03
 - B04
 - NDVI


In [8]:
variaveis_disponiveis = tuple(variavel for variavel in VARIAVEIS if any(variavel in cena.assets for cena in cenas))

if not variaveis_disponiveis:
    raise ValueError('Nenhuma variável Sentinel-2 disponível no período selecionado.')

gdf = itens_para_geodataframe(cenas, variaveis=variaveis_disponiveis)
gdf = gdf[gdf.geometry.notna()].copy()
print(f'Registros: {len(gdf)} | Variáveis: {len(variaveis_disponiveis)}')


Registros: 222 | Variáveis: 5


## GeoParquet


In [9]:
caminho_geoparquet = salvar_geoparquet(gdf, ROOT / 'data' / 'geoparquet' / f'{COLECAO}.parquet')
print('GeoParquet salvo:', caminho_geoparquet)

GeoParquet salvo: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_bdgeo\trab_final_bdgeo\data\geoparquet\S2-16D-2.parquet


## Cubo virtual


In [10]:
ds = executar_pipeline(collection=COLECAO, assets=variaveis_disponiveis)

PIPELINE SENTINEL-2

Coleção:
  S2-16D-2

Descrição:
  Sentinel-2/MSI — composto de 16 dias

Assets:
  - B01
  - B02
  - B03
  - B04
  - NDVI

ETAPA 1 — GEOPARQUET

Registros encontrados: 222

ETAPA 2 — CUBO VIRTUAL
CONSTRUÇÃO DO CUBO MULTIVARIÁVEL

Coleção: S2-16D-2

Assets solicitados:
  - B01
  - B02
  - B03
  - B04
  - NDVI

Assets disponíveis:
  [Sucesso] B01
  [Sucesso] B02
  [Sucesso] B03
  [Sucesso] B04
  [Sucesso] NDVI

------------------------------------------------------------
Asset: B01
------------------------------------------------------------


c:\Users\giuli\AppData\Local\Programs\Python\Python312\Lib\site-packages\dask\array\chunk_types.py:131: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.2)
  import scipy.sparse
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')



------------------------------------------------------------
Asset: B02
------------------------------------------------------------


ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')



------------------------------------------------------------
Asset: B03
------------------------------------------------------------

------------------------------------------------------------
Asset: B04
------------------------------------------------------------

------------------------------------------------------------
Asset: NDVI
------------------------------------------------------------


ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')
ERROR:root:Error while closing connector: ClientConnectionError('Connection lost: SSL shutdown timed out')



VALIDAÇÃO DOS CUBOS

B01:
  dimensões = {'time': 222, 'y': 10560, 'x': 10560}

B02:
  dimensões = {'time': 222, 'y': 10560, 'x': 10560}

B03:
  dimensões = {'time': 222, 'y': 10560, 'x': 10560}

B04:
  dimensões = {'time': 222, 'y': 10560, 'x': 10560}

NDVI:
  dimensões = {'time': 222, 'y': 10560, 'x': 10560}

COMBINANDO VARIÁVEIS

Cubo multivariável criado.

<xarray.Dataset> Size: 248GB
Dimensions:  (time: 222, y: 10560, x: 10560)
Coordinates:
  * time     (time) datetime64[ns] 2kB 2017-01-01 2017-01-17 ... 2026-08-13
  * y        (y) float64 84kB 8.786e+06 8.786e+06 ... 8.68e+06 8.68e+06
  * x        (x) float64 84kB 5.792e+06 5.792e+06 ... 5.898e+06 5.898e+06
Data variables:
    B01      (time, y, x) int16 50GB ManifestArray<shape=(222, 10560, 10560),...
    B02      (time, y, x) int16 50GB ManifestArray<shape=(222, 10560, 10560),...
    B03      (time, y, x) int16 50GB ManifestArray<shape=(222, 10560, 10560),...
    B04      (time, y, x) int16 50GB ManifestArray<shape=(222, 10560,

In [11]:
validacao = validar_cubo(ds)
print('Dimensões:', validacao['dimensoes'])
print('Variáveis:', validacao['variaveis'])
print('Virtual:', validacao['virtual'])


Dimensões: {'time': 222, 'y': 10560, 'x': 10560}
Variáveis: ['B01', 'B02', 'B03', 'B04', 'NDVI']
Virtual: {'B01': {'tipo': 'ManifestArray', 'virtual': True}, 'B02': {'tipo': 'ManifestArray', 'virtual': True}, 'B03': {'tipo': 'ManifestArray', 'virtual': True}, 'B04': {'tipo': 'ManifestArray', 'virtual': True}, 'NDVI': {'tipo': 'ManifestArray', 'virtual': True}}


## Icechunk


In [14]:
repo = criar_repositorio(ROOT / "data" / "icechunk_repo")

CRIANDO REPOSITÓRIO ICECHUNK

Repositório: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_bdgeo\trab_final_bdgeo\data\icechunk_repo

Container virtual: https://data.inpe.br/bdc/data/

[Sucesso] Repositório criado.


In [15]:
commit_id = salvar_dataset_virtual(ds, repo,)
print("Commit:", commit_id)


GRAVANDO CUBO VIRTUAL NO ICECHUNK

Variáveis: ['B01', 'B02', 'B03', 'B04', 'NDVI']
Dimensões: {'time': 222, 'y': 10560, 'x': 10560}
Referências virtuais: 489510
Tamanho lógico: 15,835,056 bytes

Escrevendo referências virtuais...
[Sucesso] Referências gravadas.

Realizando commit...

[Sucesso] Commit: V9QQT384S7YCHHHXZSTG
Commit: V9QQT384S7YCHHHXZSTG
